# CertVIC Linux CPython 3.10 wheelhouse provisioning
Provisioning-only: internet must be ON and accelerator OFF. This notebook creates no scientific evidence.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
WORK = Path('/kaggle/working/certvic_provisioning')
CODE = Path('/kaggle/input/certvic-code-bundle/certvic_code_bundle.zip')
TOOLS = Path('/kaggle/input/certvic-execution-tools-bundle/certvic_execution_tools_bundle.zip')
CONFIGS = Path('/kaggle/input/certvic-configs-bundle/certvic_configs_bundle.zip')
for archive in (CODE, TOOLS, CONFIGS):
    if not archive.is_file(): raise FileNotFoundError(archive)
    with zipfile.ZipFile(archive) as z: z.extractall(WORK)
os.chdir(WORK)
print({'status': 'PROVISIONING_INPUTS_READY', 'paper_evidence': False})


In [ ]:
WHEELS = Path('/kaggle/working/linux_cp310_wheels')
OUT1 = Path('/kaggle/working/certvic_offline_wheelhouse.zip')
OUT2 = Path('/kaggle/working/certvic_offline_wheelhouse.rebuild.zip')
base = [sys.executable, '-m', 'certvic.cvpr.wheelhouse_builder', '--mode', 'KAGGLE_PROVISIONING_BUILD', '--wheel-root', str(WHEELS), '--requirements-root', 'requirements']
subprocess.run(base + ['--output', str(OUT1), '--provision'], check=True)
subprocess.run(base + ['--output', str(OUT2)], check=True)
import hashlib
def sha(p):
    digest = hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()
if sha(OUT1) != sha(OUT2): raise RuntimeError('wheelhouse rebuild is not byte-identical')
OUT2.unlink()
print({'status': 'WHEELHOUSE_BUILT_DETERMINISTIC', 'path': str(OUT1), 'sha256': sha(OUT1), 'size': OUT1.stat().st_size, 'paper_evidence': False})
